# arc3-servebench27 — five-config vLLM serving sweep
Which V31 flag carries the 1.4→2.66 gap? Load-gen at concurrency 28, no games. Boot cells verbatim from the proven public notebook.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/taaf-duck-qwen38-serving-v1", "driessmit1/arc3-vllm-h100-wheelhouse-v3"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")


# V22 only modifies the vLLM launch flags inside the bundled setup command.
# The target model owns native MTP tensors and the original checkpoint validation
# already verifies that mtp.* tensors are mounted.
def _v31_serving_commands(command: str) -> tuple[str, str, bool]:
    if "vllm.entrypoints.openai.api_server" not in command:
        return command, command, False

    source_block = """        '--kv-cache-dtype',
        'fp8',
    ]"""

    v22_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
    ]"""

    v31_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
        '--no-enable-chunked-prefill',
    ]"""

    if source_block not in command:
        print("V31: known source serving block not found; leaving it unchanged.", flush=True)
        return command, command, False

    return (
        command.replace(source_block, v31_block, 1),
        command.replace(source_block, v22_block, 1),
        True,
    )


def _v22_cleanup_partial_server() -> None:
    import signal

    pid_path = WORKING_DIR / "vllm-openai-server.pid"
    if not pid_path.exists():
        return
    try:
        pid = int(pid_path.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V22: partial-server cleanup warning: {exc!r}", flush=True)
    finally:
        pid_path.unlink(missing_ok=True)


# Solver setup commands run before the benchmark loads.
# Primary = exact V22 MTP3 stack + no-chunked-prefill.
# Fallback = exact MTP3+async serving that produced 2.66 LB.
env = _command_env()
for original_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    primary_command, v22_command, optimized = _v31_serving_commands(original_command)

    if not optimized:
        print(f"taaf.kaggle: setup command: {original_command}", flush=True)
        subprocess.run(original_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    else:
        print("V31: launching MTP3 + async + FP8 KV + no-chunked-prefill.", flush=True)
        try:
            subprocess.run(primary_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
        except subprocess.CalledProcessError as exc:
            print(
                f"V31: primary startup failed ({exc!r}); "
                "falling back to exact V22 MTP3+async.",
                flush=True,
            )
            _v22_cleanup_partial_server()
            subprocess.run(v22_command, shell=True, check=True, cwd=WORKING_DIR, env=env)

    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# ---- V31 runtime resilience ----
import threading as _v31_threading
import urllib.request as _v31_urllib

_V31_URL = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
_V31_PID = WORKING_DIR / "vllm-openai-server.pid"
_V31_LOG = WORKING_DIR / "vllm-openai-server.log"
_V31_STOP = _v31_threading.Event()
_V31_THREAD = None
_V31_LOCK = _v31_threading.Lock()


def _v31_healthy(timeout: float = 4.0) -> bool:
    try:
        with _v31_urllib.urlopen(f"{_V31_URL}/models", timeout=timeout) as response:
            return 200 <= int(response.status) < 500
    except Exception:
        return False


def _v31_wait_healthy(timeout: float = 480.0) -> bool:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if _v31_healthy(5.0):
            return True
        time.sleep(5)
    return False


def _v31_kill() -> None:
    if not _V31_PID.exists():
        return
    try:
        pid = int(_V31_PID.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V31 cleanup warning: {exc!r}", flush=True)
    finally:
        _V31_PID.unlink(missing_ok=True)


def _v31_spawn(no_chunked: bool) -> None:
    provenance_file = WORKING_DIR / "qwen38-model-provenance.json"
    provenance = json.loads(provenance_file.read_text(encoding="utf-8"))
    model_path = str(provenance["model_path"])

    site_packages = WORKING_DIR / "vllm-site-packages"
    child_env = os.environ.copy()
    current_pp = child_env.get("PYTHONPATH", "")
    if str(site_packages) not in current_pp.split(os.pathsep):
        child_env["PYTHONPATH"] = (
            str(site_packages)
            if not current_pp
            else str(site_packages) + os.pathsep + current_pp
        )
    child_env.update({
        "USE_TF": "0",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_TORCHVISION": "1",
        "VLLM_NO_USAGE_STATS": "1",
    })

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--served-model-name", os.environ.get("INFERENCE_ANALYZER_MODEL", "Qwen/Qwen3.8-27B-FP8"),
        "--host", "127.0.0.1",
        "--port", "1234",
        "--tensor-parallel-size", "1",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--generation-config", "vllm",
        "--enable-prefix-caching",
        "--default-chat-template-kwargs", '{"preserve_thinking": true}',
        "--reasoning-parser", "qwen3",
        "--max-model-len", "262144",
        "--kv-cache-dtype", "fp8",
        "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
        "--async-scheduling",
    ]
    if no_chunked:
        cmd.append("--no-enable-chunked-prefill")

    # Prevent unbounded log growth across restarts.
    if _V31_LOG.exists():
        previous = WORKING_DIR / "vllm-openai-server.previous.log"
        try:
            previous.unlink(missing_ok=True)
            _V31_LOG.replace(previous)
        except OSError:
            pass

    log_handle = _V31_LOG.open("w", encoding="utf-8")
    process = subprocess.Popen(
        cmd,
        env=child_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    _V31_PID.write_text(str(process.pid), encoding="utf-8")

    if not _v31_wait_healthy():
        raise RuntimeError("restarted vLLM did not become healthy in time")


def _v31_recover() -> bool:
    with _V31_LOCK:
        if _v31_healthy():
            return True

        print("V31 watchdog: recovering MTP3/no-chunk server.", flush=True)
        _v31_kill()
        try:
            _v31_spawn(no_chunked=True)
            print("V31 watchdog: primary recovery succeeded.", flush=True)
            return True
        except Exception as first:
            print(f"V31 watchdog: primary recovery failed: {first!r}", flush=True)

        _v31_kill()
        try:
            _v31_spawn(no_chunked=False)
            print("V31 watchdog: exact V22 MTP3 recovery succeeded.", flush=True)
            return True
        except Exception as second:
            print(f"V31 watchdog: fallback recovery failed: {second!r}", flush=True)
            _v31_kill()
            return False


def _v31_start_watchdog(solver) -> None:
    global _V31_THREAD
    _V31_STOP.clear()

    def worker():
        failures = 0
        while not _V31_STOP.wait(15):
            if _v31_healthy():
                failures = 0
                continue

            failures += 1
            print(f"V31 watchdog: health failure {failures}/3", flush=True)
            if failures < 3:
                continue

            if _v31_recover():
                failures = 0
                continue

            print(
                "V31 watchdog: server unrecoverable; stopping solver to preserve partial score.",
                flush=True,
            )
            stop_event = getattr(solver, "_stop_event", None)
            if stop_event is not None:
                stop_event.set()
            return

    _V31_THREAD = _v31_threading.Thread(
        target=worker,
        name="v31-watchdog",
        daemon=True,
    )
    _V31_THREAD.start()


def _v31_stop_watchdog() -> None:
    _V31_STOP.set()
    if _V31_THREAD is not None:
        _V31_THREAD.join(timeout=5)


if not _v31_healthy(10):
    raise RuntimeError("V31 preflight: local vLLM API is not healthy.")
print("V31 preflight: local vLLM API healthy.", flush=True)



In [ ]:
# ==== servebench: five-config vLLM sweep, load-gen at concurrency 28 ====
import json as _json
import os
import signal
import socket
import statistics
import subprocess
import sys
import threading
import time
import urllib.request
from pathlib import Path

BENCH_START = time.time()
MAX_WALL_S = 4.0 * 3600          # hard budget for the whole sweep
WARMUP_S = 60
MEASURE_S = 420
CONCURRENCY = 28
PORT = 1234
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
RESULTS_PATH = Path("/kaggle/working/servebench_results.json")

_prov = _json.loads((WORKING_DIR / "qwen38-model-provenance.json").read_text())
MODEL_PATH = _prov["model_path"]
SERVED = os.environ.get("INFERENCE_ANALYZER_MODEL", "Qwen/Qwen3.8-27B-FP8")
SITE_PACKAGES = WORKING_DIR / "vllm-site-packages"

BASE_ARGS = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_PATH, "--served-model-name", SERVED,
    "--host", "127.0.0.1", "--port", str(PORT),
    "--tensor-parallel-size", "1", "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder", "--generation-config", "vllm",
    "--enable-prefix-caching",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
]
MTP = ["--speculative-config", '{"method":"mtp","num_speculative_tokens":3}']
CONFIGS = [
    ("A_v12_baseline", ["--max-model-len", "65536"]),
    ("B_fp8kv_262k", ["--max-model-len", "262144", "--kv-cache-dtype", "fp8"]),
    ("C_plus_mtp3", ["--max-model-len", "262144", "--kv-cache-dtype", "fp8", *MTP]),
    ("D_plus_async", ["--max-model-len", "262144", "--kv-cache-dtype", "fp8", *MTP,
                      "--async-scheduling"]),
    ("E_plus_nochunk", ["--max-model-len", "262144", "--kv-cache-dtype", "fp8", *MTP,
                        "--async-scheduling", "--no-enable-chunked-prefill"]),
]

# ---- traffic model: sizes from the measured live profile (chars ~ 3.3/token)
GRID_ROW = " ".join(str((i * 7) % 10) for i in range(64))
def make_payload(tokens: int, salt: int) -> str:
    rows = max(1, int(tokens * 3.3) // (len(GRID_ROW) + 1))
    return f"turn-salt:{salt}\n" + "\n".join(
        f"row{r:03d}: {GRID_ROW}" for r in range(rows))

SYSTEM_PREFIX = ("You are playing an ARC-AGI-3 grid game. Analyse the board, "
                 "maintain a world model, then act via the tool. " + make_payload(2000, 0))
BUCKETS = [("s4k", 4000, 0.20), ("m12k", 12000, 0.35),
           ("l20k", 20000, 0.30), ("xl28k", 28000, 0.15)]
TOOLS = [
    {"type": "function", "function": {"name": "python", "description": "Run python",
     "parameters": {"type": "object", "properties": {"code": {"type": "string"}},
                    "required": ["code"]}}},
    {"type": "function", "function": {"name": "action", "description": "Send game actions",
     "parameters": {"type": "object", "properties": {"actions": {"type": "array",
                    "items": {"type": "string"}}}, "required": ["actions"]}}},
]

_server = {"proc": None, "log": None}

def _port_open() -> bool:
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=2):
            return True
    except OSError:
        return False

def kill_server() -> None:
    proc = _server.get("proc")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
    # the setup-launched server has a pid file instead of a handle
    pid_path = WORKING_DIR / "vllm-openai-server.pid"
    if pid_path.exists():
        try:
            pid = int(pid_path.read_text().strip())
            for sig in (signal.SIGTERM, signal.SIGKILL):
                try:
                    os.kill(pid, sig)
                    time.sleep(3)
                except OSError:
                    break
        except Exception as exc:
            print("pid-file kill warning:", repr(exc), flush=True)
        pid_path.unlink(missing_ok=True)
    deadline = time.time() + 120
    while _port_open() and time.time() < deadline:
        time.sleep(2)
    time.sleep(5)

def spawn_server(name: str, extra: list[str]) -> Path:
    env = os.environ.copy()
    pp = env.get("PYTHONPATH", "")
    if str(SITE_PACKAGES) not in pp.split(os.pathsep):
        env["PYTHONPATH"] = str(SITE_PACKAGES) + (os.pathsep + pp if pp else "")
    env.update({"USE_TF": "0", "TRANSFORMERS_NO_TF": "1",
                "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1"})
    log_path = Path(f"/kaggle/working/servebench_{name}.log")
    proc = subprocess.Popen(BASE_ARGS + extra, env=env,
                            stdout=log_path.open("w"), stderr=subprocess.STDOUT,
                            text=True, start_new_session=True)
    _server["proc"] = proc
    _server["log"] = log_path
    return log_path

def wait_healthy(timeout: float = 600.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        proc = _server.get("proc")
        if proc is not None and proc.poll() is not None:
            return False
        try:
            with urllib.request.urlopen(f"{BASE_URL}/models", timeout=5) as r:
                if 200 <= r.status < 300:
                    return True
        except Exception:
            pass
        time.sleep(5)
    return False

def chat(messages, *, max_tokens=700, temperature=0.7, seed=None, timeout=420):
    body = {"model": SERVED, "messages": messages, "max_tokens": max_tokens,
            "temperature": temperature, "tools": TOOLS}
    if seed is not None:
        body["seed"] = seed
    req = urllib.request.Request(
        f"{BASE_URL}/chat/completions", data=_json.dumps(body).encode(),
        headers={"Content-Type": "application/json"})
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = _json.load(resp)
    usage = data.get("usage", {})
    return {"latency": time.time() - t0,
            "completion_tokens": usage.get("completion_tokens", 0),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "text": (data.get("choices") or [{}])[0].get("message", {}).get("content") or ""}

def run_load(measure_s: int) -> dict:
    stop_at = time.time() + WARMUP_S + measure_s
    window_start = time.time() + WARMUP_S
    records, errors = [], []
    lock = threading.Lock()

    def worker(wid: int):
        salt = wid * 1000
        while time.time() < stop_at:
            r = (salt * 2654435761) % 100 / 100.0
            acc, bucket = 0.0, BUCKETS[-1]
            for b in BUCKETS:
                acc += b[2]
                if r < acc:
                    bucket = b
                    break
            salt += 1
            msgs = [{"role": "system", "content": SYSTEM_PREFIX},
                    {"role": "user", "content": make_payload(bucket[1], salt)}]
            try:
                out = chat(msgs)
                done = time.time()
                with lock:
                    records.append((bucket[0], out["latency"],
                                    out["completion_tokens"], done >= window_start,
                                    done <= stop_at))
            except Exception as exc:
                with lock:
                    errors.append(repr(exc)[:200])

    threads = [threading.Thread(target=worker, args=(i,), daemon=True)
               for i in range(CONCURRENCY)]
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=WARMUP_S + measure_s + 480)
    measured = [r for r in records if r[3] and r[4]]
    by_bucket = {}
    for b, lat, ctok, *_ in measured:
        by_bucket.setdefault(b, []).append(lat)
    return {
        "measured_requests": len(measured),
        "turns_per_min": round(len(measured) / (measure_s / 60.0), 2),
        "output_tok_per_s": round(sum(r[2] for r in measured) / measure_s, 1),
        "latency_by_bucket": {b: round(statistics.mean(v), 1)
                              for b, v in sorted(by_bucket.items())},
        "errors": len(errors), "error_sample": errors[:5],
    }

def greedy_probe() -> list[str]:
    outs = []
    for i in range(8):
        msgs = [{"role": "system", "content": "Answer tersely."},
                {"role": "user", "content": f"Probe {i}: list the first {5+i} primes, "
                 "then name the action you would try first in an unknown grid game."}]
        try:
            outs.append(chat(msgs, max_tokens=96, temperature=0.0, seed=1234)["text"])
        except Exception as exc:
            outs.append(f"PROBE_ERROR {exc!r}")
    return outs

def log_evidence(log_path: Path) -> list[str]:
    pats = ("kv_cache_dtype", "KV cache", "speculative", "async_scheduling",
            "chunked_prefill", "num_speculative_tokens")
    hits = []
    try:
        for line in log_path.read_text(errors="replace").splitlines():
            if any(p.lower() in line.lower() for p in pats):
                hits.append(line.strip()[:220])
    except Exception as exc:
        hits.append(f"EVIDENCE_READ_FAIL {exc!r}")
    return hits[:30]

results = {}
for name, extra in CONFIGS:
    left = MAX_WALL_S - (time.time() - BENCH_START)
    if left < 16 * 60:
        results[name] = {"status": "SKIPPED_BUDGET", "seconds_left": int(left)}
        print(f"=== {name}: SKIPPED (budget, {int(left)}s left) ===", flush=True)
        continue
    print(f"=== {name}: relaunching server ({extra}) ===", flush=True)
    kill_server()
    log_path = spawn_server(name, extra)
    if not wait_healthy():
        tail = ""
        try:
            tail = "\n".join(log_path.read_text(errors="replace").splitlines()[-40:])
        except Exception:
            pass
        results[name] = {"status": "LAUNCH_FAIL", "log_tail": tail[-4000:]}
        print(f"=== {name}: LAUNCH_FAIL ===\n{tail[-2000:]}", flush=True)
        continue
    load = run_load(MEASURE_S)
    results[name] = {"status": "OK", "args": extra, **load,
                     "greedy": greedy_probe(), "log_evidence": log_evidence(log_path)}
    print(f"=== {name}: {load['turns_per_min']} turns/min, "
          f"{load['output_tok_per_s']} out tok/s, errors={load['errors']} ===", flush=True)
    RESULTS_PATH.write_text(_json.dumps(results, indent=1))

kill_server()
RESULTS_PATH.write_text(_json.dumps(results, indent=1))
base = results.get("A_v12_baseline", {})
print("\n==== SERVEBENCH SUMMARY ====")
for name, r in results.items():
    if r.get("status") != "OK":
        print(f"{name:16s} {r.get('status')}")
        continue
    delta = ""
    if base.get("status") == "OK" and base.get("turns_per_min"):
        delta = f" ({(r['turns_per_min'] / base['turns_per_min'] - 1) * 100:+.0f}% vs A)"
    g = sum(1 for a, b in zip(r.get("greedy", []),
                              base.get("greedy", [])) if a != b) if base.get("greedy") else "-"
    print(f"{name:16s} turns/min={r['turns_per_min']}{delta} "
          f"out_tok/s={r['output_tok_per_s']} err={r['errors']} greedy_div_vs_A={g}")
print("results ->", RESULTS_PATH)
